# Batch trial-level time-frequency dataset

This notebook generalizes the single-file demonstration to **every acquisition and online GDF recording**. It creates complete eight-second left/right motor-imagery epochs, adds one second of EEG context on each side before the Morlet transform, and crops the transformed result back to the exact trial boundaries.

The seven participants identified by the source paper as potentially problematic for neurophysiological analysis are excluded: **A4, A9, A17, A29, A41, B78, and B79**. Other experimenter comments and repository validation warnings are retained in the processing manifest rather than silently discarded. A recording is otherwise eligible only if it has a complete, balanced, internally consistent event structure.

Outputs are checkpointed per GDF so this large batch can be stopped and resumed. The neural-network tensor layout is:

`trials × 27 EEG electrodes × 23 frequencies (8–30 Hz) × 512 time points`

In [1]:
from pathlib import Path
import re
import shutil
import time

import mne
import numpy as np
import pandas as pd
from IPython.display import display

mne.set_log_level('ERROR')

def find_project_data(start=Path.cwd()):
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return parent / 'data'
    raise FileNotFoundError('Could not locate data/processed/Signals.')

DATA_ROOT = find_project_data()
SIGNALS_ROOT = DATA_ROOT / 'processed' / 'Signals'
PERFORMANCE_PATH = DATA_ROOT / 'processed' / 'Perfomances_cleaned.csv'
ISSUES_PATH = DATA_ROOT.parent / 'results' / 'reports' / 'validation_issues.csv'
OUTPUT_ROOT = DATA_ROOT / 'processed' / 'all_trials_time_frequency'
TENSOR_ROOT = OUTPUT_ROOT / 'tensors_by_file'
EVENT_ROOT = OUTPUT_ROOT / 'events_by_file'
TRIAL_ROOT = OUTPUT_ROOT / 'trials_by_file'
MANIFEST_PATH = OUTPUT_ROOT / 'processing_manifest.csv'
ALL_EVENTS_PATH = OUTPUT_ROOT / 'all_events.csv'
ALL_TRIALS_PATH = OUTPUT_ROOT / 'all_trials.csv'

for directory in (TENSOR_ROOT, EVENT_ROOT, TRIAL_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

PAPER_FLAGGED_PARTICIPANTS = {'A4', 'A9', 'A17', 'A29', 'A41', 'B78', 'B79'}
NON_EEG_CHANNELS = {'EOG1', 'EOG2', 'EOG3', 'EMGg', 'EMGd'}
FREQUENCIES_HZ = np.arange(8.0, 31.0, 1.0)
N_CYCLES = FREQUENCIES_HZ / 2.0
PADDING_SECONDS = 1.0
DECIMATION = 8
BASELINE_SECONDS_RELATIVE_TO_CUE = (-3.0, -1.0)
EXPECTED_TRIALS = 40
EXPECTED_SECONDS_PER_TRIAL = 8.0
EXPECTED_CUE_OFFSET_SECONDS = 3.0

performance = pd.read_csv(PERFORMANCE_PATH, sep=';')
issues = pd.read_csv(ISSUES_PATH)
candidate_files = sorted(
    [*SIGNALS_ROOT.rglob('*_acquisition.gdf'), *SIGNALS_ROOT.rglob('*_onlineT.gdf')],
    key=lambda path: path.as_posix().lower(),
)
if not candidate_files:
    raise FileNotFoundError(f'No acquisition/online GDF files found below {SIGNALS_ROOT}.')

estimated_output_gib = len(candidate_files) * 55 / 1024
free_gib = shutil.disk_usage(OUTPUT_ROOT).free / 1024**3
print(f'Candidate task recordings: {len(candidate_files):,}')
print(f'Approximate maximum tensor storage: {estimated_output_gib:.1f} GiB')
print(f'Free disk space: {free_gib:.1f} GiB')
if free_gib < estimated_output_gib + 5:
    raise OSError('Insufficient free space for the checkpointed TFR tensors plus a 5 GiB safety margin.')

Candidate task recordings: 520
Approximate maximum tensor storage: 27.9 GiB
Free disk space: 114.0 GiB


In [2]:
def source_metadata(path):
    relative = path.relative_to(SIGNALS_ROOT)
    participant = path.parent.name
    run_match = re.search(r'_R([1-6])_', path.name)
    if not run_match:
        raise ValueError(f'Could not parse run number from {path.name}.')
    run = int(run_match.group(1))
    return {
        'source_file': relative.as_posix(),
        'dataset': path.parent.parent.name,
        'participant': participant,
        'run': run,
        'phase': 'acquisition' if run <= 2 else 'online',
    }

def participant_comment(participant):
    rows = performance.loc[performance['SUJ_ID'].eq(participant), 'COMMENTS']
    if rows.empty or pd.isna(rows.iloc[0]):
        return ''
    return str(rows.iloc[0]).strip()

def documented_issues(metadata):
    participant = metadata['participant']
    run = metadata['run']
    matched = issues.loc[
        issues['participant_run'].fillna('').isin([participant, f'{participant}/R{run}'])
    ]
    return ' | '.join(
        f'{row.severity}:{row.check}:{row.observed_value}'
        for row in matched.itertuples(index=False)
    )

inventory = pd.DataFrame([source_metadata(path) for path in candidate_files])
inventory['paper_flagged'] = inventory['participant'].isin(PAPER_FLAGGED_PARTICIPANTS)
inventory['participant_comment'] = inventory['participant'].map(participant_comment)
inventory['documented_issues'] = [documented_issues(row) for row in inventory.to_dict('records')]

display(
    inventory.groupby(['phase', 'paper_flagged'])
    .size().rename('recordings').reset_index()
)
print('Paper-flagged participants excluded:', sorted(PAPER_FLAGGED_PARTICIPANTS))
print('Recordings with an experimenter comment:', inventory['participant_comment'].ne('').sum())
print('Recordings with a repository issue:', inventory['documented_issues'].ne('').sum())

,phase,paper_flagged,recordings
0,acquisition,False,160
1,acquisition,True,14
2,online,False,318
3,online,True,28


Paper-flagged participants excluded: ['A17', 'A29', 'A4', 'A41', 'A9', 'B78', 'B79']
Recordings with an experimenter comment: 168
Recordings with a repository issue: 13


In [3]:
EVENT_DESCRIPTIONS = {
    '32769': 'run_start', '32770': 'run_end', '768': 'trial_start',
    '786': 'fixation_cross', '33282': 'acoustic_signal',
    '769': 'left_hand_cue', '770': 'right_hand_cue',
    '781': 'feedback_start', '800': 'feedback_and_trial_end',
    '1010': 'undocumented_1010', '33281': 'undocumented_33281',
}

class IneligibleRecordingError(ValueError):
    pass

def output_paths(gdf_path):
    relative = gdf_path.relative_to(SIGNALS_ROOT).with_suffix('')
    tensor = TENSOR_ROOT / relative.parent / f'{relative.name}_trial_ersp.npz'
    events_csv = EVENT_ROOT / relative.parent / f'{relative.name}_events.csv'
    trials_csv = TRIAL_ROOT / relative.parent / f'{relative.name}_trials.csv'
    return tensor, events_csv, trials_csv

def saved_outputs_are_valid(gdf_path):
    tensor_path, events_path, trials_path = output_paths(gdf_path)
    if not all(path.is_file() for path in (tensor_path, events_path, trials_path)):
        return False
    try:
        with np.load(tensor_path, allow_pickle=False) as saved:
            if saved['X'].shape != (40, 27, 23, 512):
                return False
            if np.bincount(saved['y'], minlength=2).tolist() != [20, 20]:
                return False
        trials = pd.read_csv(trials_path)
        events_frame = pd.read_csv(events_path)
        return len(trials) == 40 and len(events_frame) > 0
    except Exception:
        return False

def atomic_csv(frame, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + '.tmp.csv')
    frame.to_csv(temporary, index=False)
    temporary.replace(destination)

In [4]:
def parse_events_and_trials(raw, metadata):
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    id_to_code = {internal_id: str(code) for code, internal_id in event_id.items()}
    event_codes = [id_to_code[event] for event in events[:, 2]]
    event_df = pd.DataFrame({
        **metadata,
        'event_number': np.arange(1, len(events) + 1),
        'sample': events[:, 0],
        'onset_seconds': events[:, 0] / raw.info['sfreq'],
        'gdf_code': event_codes,
        'event_label': [EVENT_DESCRIPTIONS.get(code, f'undocumented_{code}') for code in event_codes],
        'mne_internal_id': events[:, 2],
    })

    counts = event_df['gdf_code'].value_counts()
    required_counts = {'768': 40, '769': 20, '770': 20, '800': 40}
    wrong = {code: (int(counts.get(code, 0)), expected) for code, expected in required_counts.items() if counts.get(code, 0) != expected}
    if wrong:
        raise IneligibleRecordingError(f'incomplete/unbalanced event counts: {wrong}')

    start_rows = event_df.index[event_df['gdf_code'].eq('768')].tolist()
    trial_records = []
    for trial_number, start_row in enumerate(start_rows, start=1):
        start_sample = int(event_df.at[start_row, 'sample'])
        next_start = int(event_df.at[start_rows[trial_number], 'sample']) if trial_number < len(start_rows) else np.inf
        within = event_df.loc[event_df['sample'].ge(start_sample) & event_df['sample'].lt(next_start)]
        ends = within.loc[within['gdf_code'].eq('800')]
        cues = within.loc[within['gdf_code'].isin(['769', '770'])]
        if len(ends) != 1 or len(cues) != 1:
            raise IneligibleRecordingError(
                f'trial {trial_number} has {len(cues)} class cue(s) and {len(ends)} end marker(s)'
            )
        end_sample = int(ends.iloc[0]['sample'])
        cue_sample = int(cues.iloc[0]['sample'])
        cue_code = str(cues.iloc[0]['gdf_code'])
        trial_records.append({
            **metadata,
            'trial': trial_number,
            'start_sample': start_sample,
            'cue_sample': cue_sample,
            'end_sample': end_sample,
            'start_seconds': start_sample / raw.info['sfreq'],
            'cue_seconds': cue_sample / raw.info['sfreq'],
            'end_seconds': end_sample / raw.info['sfreq'],
            'cue_offset_seconds': (cue_sample - start_sample) / raw.info['sfreq'],
            'duration_seconds': (end_sample - start_sample) / raw.info['sfreq'],
            'gdf_cue_code': cue_code,
            'class_label': 'left' if cue_code == '769' else 'right',
        })

    trial_df = pd.DataFrame(trial_records)
    if len(trial_df) != EXPECTED_TRIALS:
        raise IneligibleRecordingError(f'expected 40 trials, found {len(trial_df)}')
    if trial_df['class_label'].value_counts().to_dict() != {'left': 20, 'right': 20}:
        raise IneligibleRecordingError('left/right classes are not balanced 20/20')
    if not np.allclose(trial_df['duration_seconds'], EXPECTED_SECONDS_PER_TRIAL):
        raise IneligibleRecordingError(f'trial durations are not uniformly {EXPECTED_SECONDS_PER_TRIAL:g} seconds')
    if not np.allclose(trial_df['cue_offset_seconds'], EXPECTED_CUE_OFFSET_SECONDS):
        raise IneligibleRecordingError(f'cue is not uniformly at +{EXPECTED_CUE_OFFSET_SECONDS:g} seconds')
    return event_df, trial_df

## Padded transformation and exact cropping

For each eight-second trial, the transform receives ten seconds: one second before the start marker, the complete trial, and one second after the end marker. The extra samples give the Morlet wavelets real neighboring EEG at the trial boundaries. Only after the transform is complete are both padding intervals removed, leaving the exact trial from −3 to just before +5 seconds relative to its cue.

In [5]:
def transform_recording(gdf_path, overwrite=False):
    metadata = source_metadata(gdf_path)
    tensor_path, events_path, trials_path = output_paths(gdf_path)

    if metadata['participant'] in PAPER_FLAGGED_PARTICIPANTS:
        return {**metadata, 'status': 'excluded_paper_flag', 'trials': 0, 'error': ''}
    if not overwrite and saved_outputs_are_valid(gdf_path):
        return {**metadata, 'status': 'skipped_valid', 'trials': 40, 'error': ''}

    raw = mne.io.read_raw_gdf(gdf_path, preload=False, verbose='ERROR')
    event_df, trial_df = parse_events_and_trials(raw, metadata)
    sfreq = float(raw.info['sfreq'])
    eeg_channels = [channel for channel in raw.ch_names if channel not in NON_EEG_CHANNELS]
    if len(eeg_channels) != 27:
        raise IneligibleRecordingError(f'expected 27 scalp EEG electrodes, found {len(eeg_channels)}')

    padding_samples = int(round(PADDING_SECONDS * sfreq))
    padded_starts = trial_df['start_sample'].to_numpy(dtype=int) - padding_samples
    padded_stops = trial_df['end_sample'].to_numpy(dtype=int) + padding_samples
    if padded_starts.min() < 0 or padded_stops.max() > raw.n_times:
        raise IneligibleRecordingError('recording does not contain the requested one-second padding around every trial')

    raw.load_data(verbose='ERROR')
    raw.filter(1.0, 40.0, picks=eeg_channels, method='fir', phase='zero', verbose='ERROR')
    padded_data = np.stack([
        raw.get_data(picks=eeg_channels, start=int(start), stop=int(stop))
        for start, stop in zip(padded_starts, padded_stops)
    ])
    if np.unique([segment.shape[-1] for segment in padded_data]).size != 1:
        raise IneligibleRecordingError('padded trial lengths are inconsistent')

    padded_power_v2 = mne.time_frequency.tfr_array_morlet(
        padded_data, sfreq=sfreq, freqs=FREQUENCIES_HZ,
        n_cycles=N_CYCLES, output='power', decim=DECIMATION,
        zero_mean=True, n_jobs=1, verbose=False,
    )
    padded_time_from_trial_start = (
        np.arange(padded_power_v2.shape[-1]) * DECIMATION / sfreq - PADDING_SECONDS
    )
    crop_mask = (
        (padded_time_from_trial_start >= 0.0)
        & (padded_time_from_trial_start < EXPECTED_SECONDS_PER_TRIAL)
    )
    power_uv2 = padded_power_v2[..., crop_mask] * 1e12
    times_relative_to_cue = padded_time_from_trial_start[crop_mask] - EXPECTED_CUE_OFFSET_SECONDS
    if power_uv2.shape != (40, 27, 23, 512):
        raise RuntimeError(f'unexpected cropped tensor shape: {power_uv2.shape}')

    baseline_mask = (
        (times_relative_to_cue >= BASELINE_SECONDS_RELATIVE_TO_CUE[0])
        & (times_relative_to_cue < BASELINE_SECONDS_RELATIVE_TO_CUE[1])
    )
    baseline_power = power_uv2[..., baseline_mask].mean(axis=-1, keepdims=True)
    tiny = np.finfo(power_uv2.dtype).tiny
    ersp_db = 10.0 * np.log10(np.maximum(power_uv2, tiny) / np.maximum(baseline_power, tiny))
    if not np.isfinite(ersp_db).all():
        raise RuntimeError('time-frequency tensor contains non-finite values')

    tensor_path.parent.mkdir(parents=True, exist_ok=True)
    tensor_tmp = tensor_path.with_name(tensor_path.name + '.tmp.npz')
    class_ids = np.where(trial_df['class_label'].eq('left'), 0, 1)
    np.savez_compressed(
        tensor_tmp,
        X=ersp_db.astype(np.float32), y=class_ids.astype(np.int8),
        class_names=np.asarray(['left', 'right']), channels=np.asarray(eeg_channels),
        frequencies_hz=FREQUENCIES_HZ,
        times_relative_to_cue_seconds=times_relative_to_cue,
        trial_numbers=trial_df['trial'].to_numpy(),
        participant=np.asarray(metadata['participant']), run=np.asarray(metadata['run']),
        phase=np.asarray(metadata['phase']), source_file=np.asarray(metadata['source_file']),
        transform_padding_seconds=np.asarray(PADDING_SECONDS),
        baseline_seconds_relative_to_cue=np.asarray(BASELINE_SECONDS_RELATIVE_TO_CUE),
    )
    tensor_tmp.replace(tensor_path)
    atomic_csv(event_df, events_path)
    atomic_csv(trial_df, trials_path)
    raw.close()
    return {**metadata, 'status': 'processed', 'trials': 40, 'error': ''}

## Smoke test

A10 Run 3 is transformed first because it has no experimenter comment, repository issue, or source-paper exclusion. This verifies the padded implementation before the full batch.

In [6]:
smoke_path = SIGNALS_ROOT / 'DATA A' / 'A10' / 'A10_R3_onlineT.gdf'
smoke_started = time.perf_counter()
smoke_result = transform_recording(smoke_path, overwrite=False)
smoke_result['seconds'] = time.perf_counter() - smoke_started
display(pd.DataFrame([smoke_result]))
assert smoke_result['status'] in {'processed', 'skipped_valid'}

,source_file,dataset,participant,run,phase,status,trials,error,seconds
0,DATA A/A10/A10_R3_onlineT.gdf,DATA A,A10,3,online,skipped_valid,40,,0.169117


## Run every candidate recording

The batch is enabled by default. Valid checkpoints are skipped. Paper-flagged participants and structurally ineligible recordings remain visible in the manifest. Unexpected processing failures are separated from eligibility exclusions and cause the cell to stop after the manifest has been saved.

In [7]:
RUN_FULL_BATCH = True
OVERWRITE = False

batch_results = []
if RUN_FULL_BATCH:
    batch_started = time.perf_counter()
    for number, gdf_path in enumerate(candidate_files, start=1):
        metadata = source_metadata(gdf_path)
        started = time.perf_counter()
        try:
            result = transform_recording(gdf_path, overwrite=OVERWRITE)
        except IneligibleRecordingError as exc:
            result = {**metadata, 'status': 'excluded_event_or_structure', 'trials': 0, 'error': str(exc)}
        except Exception as exc:
            result = {**metadata, 'status': 'failed', 'trials': 0, 'error': f'{type(exc).__name__}: {exc}'}

        result['participant_comment'] = participant_comment(metadata['participant'])
        result['documented_issues'] = documented_issues(metadata)
        result['seconds'] = time.perf_counter() - started
        batch_results.append(result)

        if number == 1 or number % 10 == 0 or result['status'] in {'failed', 'excluded_event_or_structure'} or number == len(candidate_files):
            elapsed_minutes = (time.perf_counter() - batch_started) / 60
            print(f'[{number:>3}/{len(candidate_files)}] {result["status"]:<27} {result["source_file"]} ({elapsed_minutes:.1f} min)')

    manifest = pd.DataFrame(batch_results)
    manifest.to_csv(MANIFEST_PATH, index=False)
    display(manifest['status'].value_counts().rename_axis('status').to_frame('recordings'))

    unexpected_failures = manifest.loc[manifest['status'].eq('failed')]
    if not unexpected_failures.empty:
        display(unexpected_failures)
        raise RuntimeError(f'{len(unexpected_failures)} unexpected processing failure(s); completed checkpoints are safe.')
else:
    print('Full batch disabled; only the smoke-test checkpoint was generated.')

[  1/520] excluded_event_or_structure DATA A/A1/A1_R1_acquisition.gdf (0.0 min)
[ 10/520] skipped_valid               DATA A/A10/A10_R4_onlineT.gdf (0.0 min)
[ 20/520] skipped_valid               DATA A/A12/A12_R2_acquisition.gdf (0.1 min)
[ 30/520] skipped_valid               DATA A/A13/A13_R6_onlineT.gdf (0.1 min)
[ 40/520] skipped_valid               DATA A/A15/A15_R4_onlineT.gdf (0.1 min)
[ 50/520] excluded_paper_flag         DATA A/A17/A17_R2_acquisition.gdf (0.1 min)
[ 60/520] skipped_valid               DATA A/A18/A18_R6_onlineT.gdf (0.1 min)
[ 70/520] skipped_valid               DATA A/A2/A2_R4_onlineT.gdf (0.2 min)
[ 80/520] skipped_valid               DATA A/A21/A21_R2_acquisition.gdf (0.2 min)
[ 90/520] skipped_valid               DATA A/A22/A22_R6_onlineT.gdf (0.2 min)
[100/520] skipped_valid               DATA A/A24/A24_R4_onlineT.gdf (0.2 min)
[110/520] skipped_valid               DATA A/A26/A26_R2_acquisition.gdf (0.3 min)
[120/520] skipped_valid               DATA A/A27

,recordings
status,
skipped_valid,476
excluded_paper_flag,42
excluded_event_or_structure,2


## Consolidate metadata and verify coverage

Large TFR tensors remain in per-file checkpoints. Event and trial metadata are compact enough to combine into master CSV files.

In [8]:
manifest = pd.read_csv(MANIFEST_PATH)
eligible_rows = manifest.loc[manifest['status'].isin(['processed', 'skipped_valid'])]
eligible_sources = {row.source_file for row in eligible_rows.itertuples(index=False)}

valid_paths = [
    path for path in candidate_files
    if source_metadata(path)['source_file'] in eligible_sources and saved_outputs_are_valid(path)
]
if len(valid_paths) != len(eligible_rows):
    raise RuntimeError(f'Coverage mismatch: {len(eligible_rows)} eligible manifest rows but {len(valid_paths)} valid outputs.')

all_events = pd.concat((pd.read_csv(output_paths(path)[1]) for path in valid_paths), ignore_index=True)
all_trials = pd.concat((pd.read_csv(output_paths(path)[2]) for path in valid_paths), ignore_index=True)
all_events.to_csv(ALL_EVENTS_PATH, index=False)
all_trials.to_csv(ALL_TRIALS_PATH, index=False)

assert all_trials.groupby('source_file').size().eq(40).all()
assert all_trials.groupby(['source_file', 'class_label']).size().eq(20).all()
assert all_trials['duration_seconds'].eq(8.0).all()

display(
    manifest.groupby(['phase', 'status']).size().rename('recordings').reset_index()
)
print(f'Valid transformed recordings: {len(valid_paths):,}')
print(f'Complete labeled trials: {len(all_trials):,}')
print(f'Unique participants represented: {all_trials["participant"].nunique():,}')
print(f'Manifest: {MANIFEST_PATH}')
print(f'Tensors:  {TENSOR_ROOT}')

,phase,status,recordings
0,acquisition,excluded_event_or_structure,1
1,acquisition,excluded_paper_flag,14
2,acquisition,skipped_valid,159
3,online,excluded_event_or_structure,1
4,online,excluded_paper_flag,28
5,online,skipped_valid,317


Valid transformed recordings: 476
Complete labeled trials: 19,040
Unique participants represented: 80
Manifest: /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/all_trials_time_frequency/processing_manifest.csv
Tensors:  /Users/aninanni/Documents/cosmos/26-the-data-miners-analysis/bci_cleaning/data/processed/all_trials_time_frequency/tensors_by_file


## Modeling handoff

Each `.npz` contains `X`, `y`, channel names, frequencies, cue-relative times, participant/run/phase identifiers, and normalization metadata. Acquisition and online phases must remain distinguishable during modeling. Train/validation/test splits should be grouped by participant; trial-level random splitting would leak participant-specific EEG patterns.

This notebook intentionally implements only dataset generalization and padded transformation. Trial-level artifact rejection is a separate subsequent step and has not been silently introduced here.